In [ ]:
# On your local machine
!pip install label-studio -qqq
!label-studio start


/Users/mohak/.zshenv:1: bad assignment


Config for label studio

In [1]:
import json
import glob
import re

files = sorted(glob.glob('../annotations/paper_*.json'))
tasks = []

def safe_str(val):
    if val is None:
        return "N/A"
    if isinstance(val, (dict, list)):
        return json.dumps(val)
    return str(val)

def escape_html(s):
    return str(s).replace('&','&amp;').replace('<','&lt;').replace('>','&gt;').replace('"','&quot;').replace("'","&#39;")

def build_highlighted_text(paper_text, relationships):
    """Highlight all unique taxon names as keywords in the paper text."""
    # Collect unique taxon names, longest first to avoid partial replacements
    taxon_names = list(set(
        r.get('taxon_name', '') for r in relationships
        if r.get('taxon_name')
    ))
    taxon_names.sort(key=len, reverse=True)

    # Escape HTML first, then do keyword highlighting
    text = escape_html(paper_text)

    for taxon in taxon_names:
        escaped_taxon = escape_html(taxon)
        # Skip if already wrapped to avoid double-marking
        if '<mark class="taxon-mark">' + escaped_taxon in text:
            continue
        pattern = re.compile(re.escape(escaped_taxon), re.IGNORECASE)
        text = pattern.sub('<mark class="taxon-mark">' + escaped_taxon + '</mark>', text, count=5)

    return text

def build_rel_cards(relationships):
    cards = []
    for rel in relationships:
        rel_id = rel.get('id', '')
        meta = rel.get('_annotation_metadata', {})
        verbatim = meta.get('verbatim_evidence', '') or ''
        direction = safe_str(rel.get('direction', ''))
        dir_color = '#22c55e' if direction == 'increased' else '#ef4444' if direction == 'decreased' else '#94a3b8'
        dir_arrow = '↑' if direction == 'increased' else '↓' if direction == 'decreased' else '?'

        card = (
            '<div class="rel-card" id="card-' + rel_id + '">'
            '<div class="card-header">'
            '<span class="rel-id">' + escape_html(rel_id) + '</span>'
            '<span class="taxon-badge">' + escape_html(safe_str(rel.get('taxon_name'))) + '</span>'
            '<span class="level-badge">' + escape_html(safe_str(rel.get('taxonomic_level'))) + '</span>'
            '<span class="dir-badge" style="background:' + dir_color + '22;color:' + dir_color + ';border:1px solid ' + dir_color + '44">' + dir_arrow + ' ' + escape_html(direction) + '</span>'
            '<span class="pval">p=' + escape_html(safe_str(rel.get('p_value'))) + '</span>'
            '</div>'
            '<div class="verbatim-box">📌 <em>' + (escape_html(verbatim) if verbatim else 'No verbatim evidence recorded') + '</em></div>'
            '<div class="review-notes">🗒️ ' + escape_html(safe_str(meta.get('review_notes', ''))) + '</div>'
            '<div class="verdict-row">'
            '<label><input type="radio" name="v-' + rel_id + '" value="keep" onchange="saveVerdict(\'' + rel_id + '\',\'keep\')"> ✅ Keep</label>'
            '<label><input type="radio" name="v-' + rel_id + '" value="edit" onchange="saveVerdict(\'' + rel_id + '\',\'edit\')"> ✏️ Keep w/ edits</label>'
            '<label><input type="radio" name="v-' + rel_id + '" value="reject" onchange="saveVerdict(\'' + rel_id + '\',\'reject\')"> ❌ Reject</label>'
            '</div>'
            '<textarea class="correction-box" id="note-' + rel_id + '" placeholder="Corrections or notes..."></textarea>'
            '</div>'
        )
        cards.append(card)
    return '\n'.join(cards)

CSS = """
* { box-sizing: border-box; margin: 0; padding: 0; }
body { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif; font-size: 13px; background: #f8fafc; }
.container { display: flex; height: 100vh; overflow: hidden; }
.paper-panel { flex: 1.1; overflow-y: scroll; padding: 20px; background: #fff; border-right: 2px solid #e2e8f0; line-height: 1.75; color: #1e293b; }
.paper-panel h2 { font-size: 15px; font-weight: 700; margin-bottom: 16px; color: #0f172a; border-bottom: 2px solid #e2e8f0; padding-bottom: 10px; }
.paper-text { white-space: pre-wrap; word-break: break-word; }
mark.taxon-mark { background: #dbeafe; color: #1e40af; border-radius: 3px; padding: 1px 2px; font-weight: 600; }
.rel-panel { flex: 1; overflow-y: scroll; padding: 16px; background: #f8fafc; }
.rel-panel h2 { font-size: 14px; font-weight: 700; margin-bottom: 12px; color: #0f172a; display: flex; align-items: center; gap: 8px; }
.progress-bar { background: #e2e8f0; border-radius: 99px; height: 6px; margin-bottom: 4px; }
.progress-fill { background: #6366f1; border-radius: 99px; height: 6px; transition: width 0.3s; }
.progress-label { font-size: 11px; color: #64748b; margin-bottom: 12px; }
.rel-card { background: white; border: 2px solid #e2e8f0; border-radius: 10px; padding: 14px; margin-bottom: 10px; transition: border-color 0.2s, box-shadow 0.2s; }
.rel-card:hover { border-color: #a5b4fc; box-shadow: 0 2px 8px rgba(99,102,241,0.1); }
.rel-card.verdicted-keep { border-left: 4px solid #22c55e; }
.rel-card.verdicted-edit { border-left: 4px solid #f59e0b; }
.rel-card.verdicted-reject { border-left: 4px solid #ef4444; }
.card-header { display: flex; flex-wrap: wrap; gap: 6px; align-items: center; margin-bottom: 8px; }
.rel-id { font-size: 10px; color: #94a3b8; font-family: monospace; }
.taxon-badge { background: #eff6ff; color: #1d4ed8; border: 1px solid #bfdbfe; border-radius: 4px; padding: 2px 7px; font-weight: 600; font-size: 12px; }
.level-badge { background: #f0fdf4; color: #15803d; border: 1px solid #bbf7d0; border-radius: 4px; padding: 2px 7px; font-size: 11px; }
.dir-badge { border-radius: 4px; padding: 2px 7px; font-size: 11px; font-weight: 600; }
.pval { font-size: 11px; color: #64748b; margin-left: auto; font-family: monospace; }
.verbatim-box { background: #fffde7; border-left: 3px solid #fbbf24; padding: 8px 10px; border-radius: 4px; font-size: 12px; color: #1e293b; margin-bottom: 8px; line-height: 1.5; }
.review-notes { background: #f8fafc; border-radius: 4px; padding: 8px 10px; font-size: 11px; color: #475569; margin-bottom: 10px; line-height: 1.5; }
.verdict-row { display: flex; gap: 16px; margin-bottom: 8px; }
.verdict-row label { display: flex; align-items: center; gap: 5px; cursor: pointer; font-size: 12px; font-weight: 500; }
.verdict-row input[type=radio] { cursor: pointer; }
.correction-box { width: 100%; border: 1px solid #e2e8f0; border-radius: 6px; padding: 6px 8px; font-size: 12px; font-family: inherit; resize: vertical; min-height: 50px; color: #374151; }
.correction-box:focus { outline: none; border-color: #6366f1; }
.submit-section { margin-top: 16px; padding: 14px; background: white; border: 2px solid #e2e8f0; border-radius: 10px; }
.submit-section h3 { font-size: 13px; font-weight: 600; margin-bottom: 10px; color: #0f172a; }
"""

JS = """
const verdicts = {};

function saveVerdict(relId, verdict) {
  if (verdict) verdicts[relId] = verdict;
  const card = document.getElementById('card-' + relId);
  card.classList.remove('verdicted-keep','verdicted-edit','verdicted-reject');
  card.classList.add('verdicted-' + verdicts[relId]);
  updateProgress();
}

function updateProgress() {
  const cards = document.querySelectorAll('.rel-card');
  const done = Object.keys(verdicts).length;
  const total = cards.length;
  const pct = total > 0 ? (done/total*100) : 0;
  document.getElementById('progressFill').style.width = pct + '%';
  document.getElementById('progressLabel').textContent = done + ' / ' + total + ' verdicted';
}

function buildExportJSON() {
  const cards = document.querySelectorAll('.rel-card');
  const results = [];
  cards.forEach(card => {
    const relId = card.id.replace('card-', '');
    const selected = card.querySelector('input[type=radio]:checked');
    const noteBox = document.getElementById('note-' + relId);
    const taxonEl = card.querySelector('.taxon-badge');
    const levelEl = card.querySelector('.level-badge');
    const dirEl = card.querySelector('.dir-badge');
    const pvalEl = card.querySelector('.pval');
    results.push({
      rel_id: relId,
      verdict: selected ? selected.value : null,
      corrections: noteBox ? noteBox.value.trim() : '',
      taxon_name: taxonEl ? taxonEl.textContent.trim() : '',
      taxonomic_level: levelEl ? levelEl.textContent.trim() : '',
      direction: dirEl ? dirEl.textContent.replace(/[\\u2191\\u2193?]/g, '').trim() : '',
      p_value: pvalEl ? pvalEl.textContent.replace('p=', '').trim() : ''
    });
  });
  const paperNotes = document.getElementById('paperNotes');
  return JSON.stringify({
    relationships: results,
    paper_notes: paperNotes ? paperNotes.value.trim() : '',
    verdicted_count: Object.keys(verdicts).length,
    total_count: cards.length,
    timestamp: new Date().toISOString()
  }, null, 2);
}

function copyAnnotations() {
  const jsonStr = buildExportJSON();
  navigator.clipboard.writeText(jsonStr).then(() => {
    const btn = document.getElementById('copyBtn');
    btn.textContent = 'Copied! Paste into text box below, then Submit.';
    btn.style.background = '#22c55e';
    setTimeout(() => { btn.textContent = 'Copy Annotations JSON'; btn.style.background = '#6366f1'; }, 4000);
  });
}
"""

for fp in files:
    with open(fp) as f:
        d = json.load(f)

    paper_id = safe_str(d.get('paper_id'))
    paper_title = safe_str(d.get('paper_title'))
    paper_text = d.get('paper_text', '') or ''
    relationships = d.get('relationships', [])
    final_count = d.get('extraction_stats', {}).get('final_count', 0)

    highlighted_text = build_highlighted_text(paper_text, relationships)
    rel_cards = build_rel_cards(relationships)

    html_content = (
        '<!DOCTYPE html><html><head><style>' + CSS + '</style></head><body>'
        '<div class="container">'
        '<div class="paper-panel">'
        '<h2>📄 ' + escape_html(paper_title) + '</h2>'
        '<p style="font-size:11px;color:#94a3b8;margin-bottom:12px;">💡 Taxon names highlighted in blue. Use Ctrl+F to find specific text.</p>'
        '<div class="paper-text">' + highlighted_text + '</div>'
        '</div>'
        '<div class="rel-panel">'
        '<h2>🔬 Relationships <span style="background:#e0e7ff;color:#4338ca;border-radius:99px;padding:2px 10px;font-size:12px">' + str(final_count) + ' total</span></h2>'
        '<div class="progress-bar"><div class="progress-fill" id="progressFill" style="width:0%"></div></div>'
        '<div class="progress-label" id="progressLabel">0 / ' + str(final_count) + ' verdicted</div>'
        + rel_cards +
        '<div class="submit-section"><h3>📋 Paper-level Notes</h3>'
        '<textarea class="correction-box" id="paperNotes" rows="3" placeholder="Overall notes about this paper\'s extraction quality..."></textarea>'
        '<button id="copyBtn" onclick="copyAnnotations()" style="width:100%;padding:12px;background:#6366f1;color:white;border:none;border-radius:8px;font-size:14px;font-weight:600;cursor:pointer;margin-top:12px;">Copy Annotations JSON</button>'
        '</div></div></div>'
        '<script>' + JS + '</script>'
        '</body></html>'
    )

    tasks.append({
        "data": {
            "paper_id": paper_id,
            "paper_title": paper_title,
            "content": html_content,
            "rel_count": str(final_count)
        }
    })

with open('ls_import_per_paper.json', 'w') as f:
    json.dump(tasks, f, indent=2)

print(f"✓ {len(files)} papers → {len(tasks)} tasks")
print(f"Output: ls_import_per_paper.json")

✓ 30 papers → 30 tasks
Output: ls_import_per_paper.json
